In [10]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import xml.etree.ElementTree as ET
import numpy as np


# Scraping eu-site
**What should be included:**
- Names (rappertour and shadow)
- Field
- Political party
- Document text

Names, field, document code and political party can be scraped from: https://oeil.secure.europarl.europa.eu/oeil/en/search?fullText.mode=EXACT_WORD&term=9th+term+2019+-+2024

Shadow rapporteur from: https://oeil.secure.europarl.europa.eu/oeil/en/procedure-file?
- The document code should be inputtet, and can be fetched from the previous

Context of the document comes from: https://oeil.secure.europarl.europa.eu/oeil/en/document-summary?id=1785947
- Id can be found by scraping the above web page, stored under Legislative proposal


In [11]:
#Finding out how it can be accesed
link = "https://oeil.secure.europarl.europa.eu/oeil/en/search?fullText.mode=EXACT_WORD&reference.type=legAct&reference.initialType=legAct&term=9th+term+2019+-+2024&resultsOnly=true"
r = requests.get(link)
soup = BeautifulSoup(r.content)
names = soup.find_all("span", class_="erpl_document-subtitle-author")
rapporteurs = [name.text.strip() for name in names]
rapporteurs_out = [", ".join(rapporteurs)]
rapporteurs_out

['CUNHA Paulo (EPP), GONZÁLEZ CASARES Nicolás (S&D), KELLER Fabienne (Renew), GREGOROVÁ Markéta (Greens/EFA), KALNIETE Sandra (EPP), SIPPEL Birgit (S&D), NEMEC Matjaž (S&D), RESSLER Karlo (EPP), PICULA Tonino (S&D), MANDERS Antonius (EPP), KUHNKE Alice (Greens/EFA), AGUILERA Clara (S&D), BALLARÍN CEREZA Laura (S&D), CAVAZZINI Anna (Greens/EFA), SINČIĆ Ivan Vilibor (NI), VOSS Axel (EPP), OETJEN Jan-Christoph (Renew), OETJEN Jan-Christoph (Renew), LÓPEZ AGUILAR Juan Fernando (S&D), SINČIĆ Ivan Vilibor (NI), ĎURIŠ NICHOLSONOVÁ Lucia (Renew), BIELAN Adam (ECR), MORTLER Marlene (EPP), GUERREIRO Francisco (Greens/EFA), VAN OVERTVELDT Johan (ECR), NEMEC Matjaž (S&D), HAUTALA Heidi (Greens/EFA), BORCHIA Paolo (ID), GAHLER Michael (EPP), GARDIAZABAL RUBIAL Eider (S&D)']

In [12]:
commiteer = soup.find_all("span", class_="erpl_badge-committee")
committee_title = [c.get("title") for c in commiteer if c.has_attr("title")]
committee_title

['Civil Liberties, Justice and Home Affairs',
 'Agriculture and Rural Development',
 'Industry, Research and Energy',
 'Civil Liberties, Justice and Home Affairs',
 'International Trade',
 'International Trade',
 'Environment, Climate and Food Safety',
 'Economic and Monetary Affairs',
 'Civil Liberties, Justice and Home Affairs',
 'Civil Liberties, Justice and Home Affairs',
 'Foreign Affairs',
 'Budgets',
 'Employment and Social Affairs',
 'Civil Liberties, Justice and Home Affairs',
 'Agriculture and Rural Development',
 'Internal Market and Consumer Protection',
 'Internal Market and Consumer Protection',
 'Environment, Public Health and Food Safety',
 'Legal Affairs',
 'Transport and Tourism',
 'Transport and Tourism',
 'Civil Liberties, Justice and Home Affairs',
 'Environment, Public Health and Food Safety',
 'Employment and Social Affairs',
 'Internal Market and Consumer Protection',
 'Environment, Public Health and Food Safety',
 'Fisheries',
 'Economic and Monetary Affairs',


In [13]:
# URL  - her kun sorteret for emne Agriculture, hvis datasættet skal udvides kan man bare søge efter andre emner og så gemme i seperate datasæt
# som til sidst kan sættes sammen (det tror jeg er nemmest)
url = "https://oeil.secure.europarl.europa.eu/oeil/en/search/export/XML?fullText.mode=EXACT_WORD&term=9th+term+2019+-+2024&subject=3.10+Agricultural+policy+and+economies+&resultsOnly=true"


response = requests.get(url)
response.raise_for_status()

root = ET.fromstring(response.content)


data = []

for item in root.find("items").findall("item"):
    reference = item.findtext("reference", default="")
    title = item.findtext("title", default="")
    
    # Extract all rapporteurs 
    rapporteurs_elem = item.find("rapporteur")
    if rapporteurs_elem is not None:
        rapporteurs = [r.text for r in rapporteurs_elem.findall("rapporteur") if r.text]
    else:
        rapporteurs = ['NaN']

    rapporteurs_str = ", ".join(rapporteurs) if rapporteurs else ""

    data.append({
        "document_id": reference,
        "title": title,
        "rapporteurs": rapporteurs_str,
    })

df = pd.DataFrame(data)
print(df.head())


      document_id                                              title  \
0  2024/0144(COD)  Economic accounts for agriculture in the Union...   
1  2024/0073(COD)  Common Agricultural Policy (CAP): good agricul...   
2  2024/0030(COD)  Equivalence of field inspections carried out i...   
3  2024/0027(COD)  Granting equivalence with EU requirements to M...   
4  2023/0448(COD)  Protection of animals during transport and rel...   

                                  rapporteurs  
0                                              
1                                              
2                                              
3                   VRECIONOVÁ Veronika (ECR)  
4  BUDA Daniel (EPP), METZ Tilly (Greens/EFA)  


In [14]:
df['rapporteurs'] = df['rapporteurs'].replace('', np.nan) 
df

,document_id,title,rapporteurs
0,2024/0144(COD),Economic accounts for agriculture in the Union...,NaN
1,2024/0073(COD),Common Agricultural Policy (CAP): good agricul...,NaN
2,2024/0030(COD),Equivalence of field inspections carried out i...,NaN
3,2024/0027(COD),Granting equivalence with EU requirements to M...,VRECIONOVÁ Veronika (ECR)
4,2023/0448(COD),Protection of animals during transport and rel...,"BUDA Daniel (EPP), METZ Tilly (Greens/EFA)"
...,...,...,...
380,2020/2734(RPS),Resolution on the draft Commission regulation ...,NaN
381,2019/2776(RPS),Resolution on the draft Commission regulation ...,"ANDRIEU Eric (S&D), HOJSÍK Martin (Renew), EIC..."
382,COM(2024)0225,Force majeure and exceptional circumstances in...,NaN
383,COM(2024)0194,International Olive Council (IOC): two methods...,NaN


In [15]:
#removing nan
df_agri = df.dropna()
df_agri.reset_index(drop=True, inplace= True)
df_agri

,document_id,title,rapporteurs
0,2024/0027(COD),Granting equivalence with EU requirements to M...,VRECIONOVÁ Veronika (ECR)
1,2023/0448(COD),Protection of animals during transport and rel...,"BUDA Daniel (EPP), METZ Tilly (Greens/EFA)"
2,2023/0447(COD),Welfare of dogs and cats and their traceability,VRECIONOVÁ Veronika (ECR)
3,2023/0413(COD),Monitoring framework for resilient European fo...,"SARGIACOMO Eric (S&D), WIESNER Emma (Renew)"
4,2023/0410(COD),Standing Forest and Forestry Expert Group,"TOVERI Pekka (EPP), WIESNER Emma (Renew)"
...,...,...,...
83,2023/2726(RPS),Commission Regulation amending Annex II to Reg...,RIVASI Michèle (Greens/EFA)
84,2021/2608(RPS),Resolution on the draft Commission regulation ...,RIVASI Michèle (Greens/EFA)
85,2020/2795(RPS),Resolution on the draft Commission regulation ...,"NOVAK Ljudmila (EPP), ANDRIEU Eric (S&D), RIVA..."
86,2020/2735(RPS),Resolution on the draft Commission regulation ...,"PIETIKÄINEN Sirpa (EPP), SCHALDEMOSE Christel ..."


# Find shadow rapporteurs and document url

In [16]:
df_test = df_agri.copy()
df_test = df_test[:1]
df_test

,document_id,title,rapporteurs
0,2024/0027(COD),Granting equivalence with EU requirements to M...,VRECIONOVÁ Veronika (ECR)


In [17]:
def get_shadow_rapporteurs(reference):
    base_url = "https://oeil.secure.europarl.europa.eu/oeil/en/procedure-file?reference="
    url = base_url + reference

    response = requests.get(url)
    if response.status_code != 200:
        print(f"Failed to fetch: {url}")
        return None

    soup = BeautifulSoup(response.content, "html.parser")

    try:
        shadow_section = soup.find("div", id="collapseShadowRapporteur")
        if not shadow_section:
            return None
        
        shadow_names = []
        for a in shadow_section.find_all("a", class_= "rapporteur"):
            span = a.find("span")
            #print("span", span)
            if span:
                shadow_names.append(span.get_text(strip=True))
                #print("shadow names:", shadow_names)

        return ", ".join(shadow_names) if shadow_names else None
    
    except Exception as e:
        print(f"Error parsing {reference}: {e}")
        return None

df_agri["shadow_rapporteurs"] = df_agri["document_id"].apply(get_shadow_rapporteurs)


/var/folders/c_/q2yk81hd0zj7jnx_b5275pzh0000gn/T/ipykernel_7561/2399249807.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_agri["shadow_rapporteurs"] = df_agri["document_id"].apply(get_shadow_rapporteurs)


In [18]:
df_agri

,document_id,title,rapporteurs,shadow_rapporteurs
0,2024/0027(COD),Granting equivalence with EU requirements to M...,VRECIONOVÁ Veronika (ECR),"BUDA Daniel (EPP), CÂRCIU Gheorghe (S&D), PENN..."
1,2023/0448(COD),Protection of animals during transport and rel...,"BUDA Daniel (EPP), METZ Tilly (Greens/EFA)","GIMÉNEZ LARRAZ Borja (EPP), VIND Marianne (S&D..."
2,2023/0447(COD),Welfare of dogs and cats and their traceability,VRECIONOVÁ Veronika (ECR),"DE MEO Salvatore (EPP), NARDELLA Dario (S&D), ..."
3,2023/0413(COD),Monitoring framework for resilient European fo...,"SARGIACOMO Eric (S&D), WIESNER Emma (Renew)","KÖHLER Stefan (EPP), BERNHUBER Alexander (EPP)..."
4,2023/0410(COD),Standing Forest and Forestry Expert Group,"TOVERI Pekka (EPP), WIESNER Emma (Renew)","BERNHUBER Alexander (EPP), TEMIDO Marta (S&D),..."
...,...,...,...,...
83,2023/2726(RPS),Commission Regulation amending Annex II to Reg...,RIVASI Michèle (Greens/EFA),None
84,2021/2608(RPS),Resolution on the draft Commission regulation ...,RIVASI Michèle (Greens/EFA),"SCHNEIDER Christine (EPP), HUITEMA Jan (Renew)"
85,2020/2795(RPS),Resolution on the draft Commission regulation ...,"NOVAK Ljudmila (EPP), ANDRIEU Eric (S&D), RIVA...",None
86,2020/2735(RPS),Resolution on the draft Commission regulation ...,"PIETIKÄINEN Sirpa (EPP), SCHALDEMOSE Christel ...",None


### **Fisheries policy**

### **Industrial policy**

In [38]:
# Industrial policy
url_indu = "https://oeil.secure.europarl.europa.eu/oeil/en/search/export/XML?fullText.mode=EXACT_WORD&term=9th+term+2019+-+2024&subject=3.40+Industrial+policy&resultsOnly=true"

response_indu = requests.get(url_indu)
response_indu.raise_for_status()

root = ET.fromstring(response_indu.content)

data_indu = []

for item in root.find("items").findall("item"):
    reference = item.findtext("reference", default="")
    title = item.findtext("title", default="")
    
    # Extract all rapporteurs 
    rapporteurs_elem_indu = item.find("rapporteur")
    if rapporteurs_elem_indu is not None:
        rapporteurs = [r.text for r in rapporteurs_elem_indu.findall("rapporteur") if r.text]
    else:
        rapporteurs = ['NaN']

    rapporteurs_str = ", ".join(rapporteurs) if rapporteurs else ""

    data_indu.append({
        "document_id": reference,
        "title": title,
        "rapporteurs": rapporteurs_str,
    })

df_indu = pd.DataFrame(data_indu)
print(df_indu.head())

      document_id                                              title  \
0  2024/0061(COD)  European Defence Industry Programme and framew...   
1  2024/0021(COD)  Gradual roll-out of Eudamed, information oblig...   
2  2023/0453(COD)  Common data platform on chemicals, establishin...   
3  2023/0455(COD)  Chemicals: re-attribution of scientific and te...   
4  2023/0454(COD)  Restriction of the use of certain hazardous su...   

                                         rapporteurs  
0  BELLAMY François-Xavier (EPP), GLUCKSMANN Raph...  
1                                                     
2                            TSIODRAS Dimitris (EPP)  
3                            TSIODRAS Dimitris (EPP)  
4                            TSIODRAS Dimitris (EPP)  


In [39]:
df_indu['rapporteurs'] = df_indu['rapporteurs'].replace('', np.nan) 
df_indu

,document_id,title,rapporteurs
0,2024/0061(COD),European Defence Industry Programme and framew...,"BELLAMY François-Xavier (EPP), GLUCKSMANN Raph..."
1,2024/0021(COD),"Gradual roll-out of Eudamed, information oblig...",NaN
2,2023/0453(COD),"Common data platform on chemicals, establishin...",TSIODRAS Dimitris (EPP)
3,2023/0455(COD),Chemicals: re-attribution of scientific and te...,TSIODRAS Dimitris (EPP)
4,2023/0454(COD),Restriction of the use of certain hazardous su...,TSIODRAS Dimitris (EPP)
...,...,...,...
241,2019/2771(DEA),Export and import of hazardous chemicals: amen...,NaN
242,2019/2763(DEA),Technical requirements for inland waterway ves...,NaN
243,2020/2898(RPS),Decision to raise no objections to the draft C...,CANFIN Pascal (Renew)
244,COM(2024)0264,Signing of the Council of Europe Framework Con...,NaN


In [40]:
#removing nan
df_indu1 = df_indu.dropna()
df_indu1.reset_index(drop=True, inplace= True)
df_indu1

,document_id,title,rapporteurs
0,2024/0061(COD),European Defence Industry Programme and framew...,"BELLAMY François-Xavier (EPP), GLUCKSMANN Raph..."
1,2023/0453(COD),"Common data platform on chemicals, establishin...",TSIODRAS Dimitris (EPP)
2,2023/0455(COD),Chemicals: re-attribution of scientific and te...,TSIODRAS Dimitris (EPP)
3,2023/0454(COD),Restriction of the use of certain hazardous su...,TSIODRAS Dimitris (EPP)
4,2023/0373(COD),Preventing plastic pellet losses to reduce mic...,LUENA César (S&D)
...,...,...,...
67,2019/2606(RSP),Objection pursuant to Rule 106: Authorisation ...,"POC Pavel (S&D), KONEČNÁ Kateřina (GUE/NGL)"
68,2019/2605(RSP),Objection pursuant to Rule 106: Authorisation ...,"POC Pavel (S&D), KONEČNÁ Kateřina (GUE/NGL)"
69,2024/2691(DEA),Definition of ‘engineered nanomaterials’,"PIETIKÄINEN Sirpa (EPP), SCHALDEMOSE Christel ..."
70,2019/2843(DEA),"Classification, labelling and packing of subst...",ZALEWSKA Anna (ECR)


#### Find shadow rapporteurs and document url

In [41]:
df_test = df_indu1.copy()
df_test = df_test[:1]
df_test

,document_id,title,rapporteurs
0,2024/0061(COD),European Defence Industry Programme and framew...,"BELLAMY François-Xavier (EPP), GLUCKSMANN Raph..."


In [42]:
def get_shadow_and_link(reference):
    base_url = "https://oeil.secure.europarl.europa.eu/oeil/en/procedure-file?reference="
    url_indu = base_url + reference

    response = requests.get(url_indu)
    if response.status_code != 200:
        print(f"Failed to fetch: {url_indu}")
        return None

    soup = BeautifulSoup(response.content, "html.parser")

    try:
        #finding shadow rapporteurs
        shadow_section = soup.find("div", id="collapseShadowRapporteur")
        if not shadow_section:
            return None
        
        shadow_names = []
        for a in shadow_section.find_all("a", class_= "rapporteur"):
            span = a.find("span")
            #print("span", span)
            if span:
                shadow_names.append(span.get_text(strip=True))
                #print("shadow names:", shadow_names)

        shadow_str = ", ".join(shadow_names) if shadow_names else None

        #finding summary link
        section6 = soup.find("div", id="section6")
        oeil_link = None  # <-- initialize here

        if section6:
            ec_span = section6.find("span", string="European Commission")
            if ec_span:
                accordion_item = ec_span.find_parent("li", class_="erpl_accordion-item")
                if accordion_item:
                    summary_links = accordion_item.find_all("a", href=True)
                    for a in summary_links:
                        href = a.get("href")
                        if href and href.startswith("/oeil"):
                            oeil_link = "https://oeil.secure.europarl.europa.eu" + href
                            break  

        return shadow_str, oeil_link



    
    except Exception as e:
        print(f"Error parsing {reference}: {e}")
        return None

df_indu1[["shadow_rapporteurs","summary_link"]] = df_indu1["document_id"].apply(
    lambda ref: pd.Series(get_shadow_and_link(ref))
)

/var/folders/c_/q2yk81hd0zj7jnx_b5275pzh0000gn/T/ipykernel_7561/2878242846.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_indu1[["shadow_rapporteurs","summary_link"]] = df_indu1["document_id"].apply(
/var/folders/c_/q2yk81hd0zj7jnx_b5275pzh0000gn/T/ipykernel_7561/2878242846.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_indu1[["shadow_rapporteurs","summary_link"]] = df_indu1["document_id"].apply(


In [43]:
df_indu1

,document_id,title,rapporteurs,shadow_rapporteurs,summary_link
0,2024/0061(COD),European Defence Industry Programme and framew...,"BELLAMY François-Xavier (EPP), GLUCKSMANN Raph...","GAHLER Michael (EPP), MANIATIS Yannis (S&D), S...",https://oeil.secure.europarl.europa.eu/oeil/en...
1,2023/0453(COD),"Common data platform on chemicals, establishin...",TSIODRAS Dimitris (EPP),"CLERGEAU Christophe (S&D), TIMGREN Beatrice (E...",https://oeil.secure.europarl.europa.eu/oeil/en...
2,2023/0455(COD),Chemicals: re-attribution of scientific and te...,TSIODRAS Dimitris (EPP),"CLERGEAU Christophe (S&D), TIMGREN Beatrice (E...",https://oeil.secure.europarl.europa.eu/oeil/en...
3,2023/0454(COD),Restriction of the use of certain hazardous su...,TSIODRAS Dimitris (EPP),"CLERGEAU Christophe (S&D), TIMGREN Beatrice (E...",https://oeil.secure.europarl.europa.eu/oeil/en...
4,2023/0373(COD),Preventing plastic pellet losses to reduce mic...,LUENA César (S&D),"SOMMEN Liesbet (EPP), BONTE Barbara (PfE), FIO...",https://oeil.secure.europarl.europa.eu/oeil/en...
...,...,...,...,...,...
67,2019/2606(RSP),Objection pursuant to Rule 106: Authorisation ...,"POC Pavel (S&D), KONEČNÁ Kateřina (GUE/NGL)",NaN,NaN
68,2019/2605(RSP),Objection pursuant to Rule 106: Authorisation ...,"POC Pavel (S&D), KONEČNÁ Kateřina (GUE/NGL)",NaN,NaN
69,2024/2691(DEA),Definition of ‘engineered nanomaterials’,"PIETIKÄINEN Sirpa (EPP), SCHALDEMOSE Christel ...",NaN,NaN
70,2019/2843(DEA),"Classification, labelling and packing of subst...",ZALEWSKA Anna (ECR),NaN,NaN


In [44]:
df_indu1.to_csv("df_industrial.csv", index=False)
print("CSV file saved successfully!")

CSV file saved successfully!


### **Enterprise policy, inter-company cooperation**

In [50]:
url_enter = "https://oeil.secure.europarl.europa.eu/oeil/en/search/export/XML?fullText.mode=EXACT_WORD&term=9th+term+2019+-+2024&subject=3.45+Enterprise+policy%2C+inter-company+cooperation&resultsOnly=true"

response = requests.get(url_enter)
response.raise_for_status()

root = ET.fromstring(response.content)

data_enter = []

for item in root.find("items").findall("item"):
    reference = item.findtext("reference", default="")
    title = item.findtext("title", default="")
    
    # Extract all rapporteurs 
    rapporteurs_elem = item.find("rapporteur")
    if rapporteurs_elem is not None:
        rapporteurs = [r.text for r in rapporteurs_elem.findall("rapporteur") if r.text]
    else:
        rapporteurs = ['NaN']

    rapporteurs_str = ", ".join(rapporteurs) if rapporteurs else ""

    data_enter.append({
        "document_id": reference,
        "title": title,
        "rapporteurs": rapporteurs_str,
    })

df_enter = pd.DataFrame(data_enter)
print(df_enter.head())

      document_id                                              title  \
0  2023/0376(COD)  Amending ADR Directive and certain other consu...   
1  2023/0375(COD)       Discontinuation of the European ODR Platform   
2  2023/0368(COD)  Company law: time limits for the adoption of s...   
3  2023/0314(COD)  Use of the Internal Market Information System ...   
4  2023/0323(COD)    Commercial transactions: combating late payment   

                   rapporteurs  
0  BALLARÍN CEREZA Laura (S&D)  
1  BALLARÍN CEREZA Laura (S&D)  
2              VOSS Axel (EPP)  
3       GEDIN Hanna (The Left)  
4          IJABS Ivars (Renew)  


In [51]:
df_enter['rapporteurs'] = df_enter['rapporteurs'].replace('', np.nan) 
df_enter

,document_id,title,rapporteurs
0,2023/0376(COD),Amending ADR Directive and certain other consu...,BALLARÍN CEREZA Laura (S&D)
1,2023/0375(COD),Discontinuation of the European ODR Platform,BALLARÍN CEREZA Laura (S&D)
2,2023/0368(COD),Company law: time limits for the adoption of s...,VOSS Axel (EPP)
3,2023/0314(COD),Use of the Internal Market Information System ...,GEDIN Hanna (The Left)
4,2023/0323(COD),Commercial transactions: combating late payment,IJABS Ivars (Renew)
...,...,...,...
87,2021/2639(DEA),"Integration of sustainability factors, risks a...",NaN
88,2020/2851(RPS),Decision to raise no objections to the draft C...,NaN
89,2020/2712(RPS),Decision to raise no objections to the draft C...,NaN
90,SWD(2024)0172,Annual report on taxation 2024 - Review of tax...,NaN


In [52]:
#removing nan
df_enter1 = df_enter.dropna()
df_enter1.reset_index(drop=True, inplace= True)
df_enter1

,document_id,title,rapporteurs
0,2023/0376(COD),Amending ADR Directive and certain other consu...,BALLARÍN CEREZA Laura (S&D)
1,2023/0375(COD),Discontinuation of the European ODR Platform,BALLARÍN CEREZA Laura (S&D)
2,2023/0368(COD),Company law: time limits for the adoption of s...,VOSS Axel (EPP)
3,2023/0314(COD),Use of the Internal Market Information System ...,GEDIN Hanna (The Left)
4,2023/0323(COD),Commercial transactions: combating late payment,IJABS Ivars (Renew)
5,2023/0315(COD),European cross-border associations,LAGODINSKY Sergey (Greens/EFA)
6,2023/0202(COD),General Data Protection Regulation: additional...,GREGOROVÁ Markéta (Greens/EFA)
7,2023/0089(COD),Company law: further expanding and upgrading t...,RADEV Emil (EPP)
8,2022/0411(COD),Making public capital markets in the Union mor...,SANT Alfred (S&D)
9,2022/0408(COD),Harmonising certain aspects of insolvency law,RADEV Emil (EPP)


#### Find shadow rapporteurs and document url

In [53]:
df_test = df_enter1.copy()
df_test = df_test[:1]
df_test

,document_id,title,rapporteurs
0,2023/0376(COD),Amending ADR Directive and certain other consu...,BALLARÍN CEREZA Laura (S&D)


In [55]:
def get_shadow_and_link(reference):
    base_url = "https://oeil.secure.europarl.europa.eu/oeil/en/procedure-file?reference="
    url_enter = base_url + reference

    response = requests.get(url_enter)
    if response.status_code != 200:
        print(f"Failed to fetch: {url_enter}")
        return None

    soup = BeautifulSoup(response.content, "html.parser")

    try:
        #finding shadow rapporteurs
        shadow_section = soup.find("div", id="collapseShadowRapporteur")
        if not shadow_section:
            return None
        
        shadow_names = []
        for a in shadow_section.find_all("a", class_= "rapporteur"):
            span = a.find("span")
            #print("span", span)
            if span:
                shadow_names.append(span.get_text(strip=True))
                #print("shadow names:", shadow_names)

        shadow_str = ", ".join(shadow_names) if shadow_names else None

        #finding summary link
        section6 = soup.find("div", id="section6")
        oeil_link = None  # <-- initialize here

        if section6:
            ec_span = section6.find("span", string="European Commission")
            if ec_span:
                accordion_item = ec_span.find_parent("li", class_="erpl_accordion-item")
                if accordion_item:
                    summary_links = accordion_item.find_all("a", href=True)
                    for a in summary_links:
                        href = a.get("href")
                        if href and href.startswith("/oeil"):
                            oeil_link = "https://oeil.secure.europarl.europa.eu" + href
                            break  

        return shadow_str, oeil_link



    
    except Exception as e:
        print(f"Error parsing {reference}: {e}")
        return None

df_enter1[["shadow_rapporteurs","summary_link"]] = df_enter1["document_id"].apply(
    lambda ref: pd.Series(get_shadow_and_link(ref))
)

/var/folders/c_/q2yk81hd0zj7jnx_b5275pzh0000gn/T/ipykernel_7561/843607443.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_enter1[["shadow_rapporteurs","summary_link"]] = df_enter1["document_id"].apply(


In [56]:
df_enter1

,document_id,title,rapporteurs,shadow_rapporteurs,summary_link
0,2023/0376(COD),Amending ADR Directive and certain other consu...,BALLARÍN CEREZA Laura (S&D),"VAIDERE Inese (EPP), TUREK Filip (PfE), PIPERE...",https://oeil.secure.europarl.europa.eu/oeil/en...
1,2023/0375(COD),Discontinuation of the European ODR Platform,BALLARÍN CEREZA Laura (S&D),"VAIDERE Inese (EPP), PIPEREA Gheorghe (ECR), Y...",https://oeil.secure.europarl.europa.eu/oeil/en...
2,2023/0368(COD),Company law: time limits for the adoption of s...,VOSS Axel (EPP),"DURAND Pascal (S&D), KARLESKIND Pierre (Renew)...",https://oeil.secure.europarl.europa.eu/oeil/en...
3,2023/0314(COD),Use of the Internal Market Information System ...,GEDIN Hanna (The Left),"AGIUS Peter (EPP), GRAPINI Maria (S&D), SCHALL...",None
4,2023/0323(COD),Commercial transactions: combating late payment,IJABS Ivars (Renew),"DOHERTY Regina (EPP), PENKOVA Tsvetelina (S&D)...",https://oeil.secure.europarl.europa.eu/oeil/en...
5,2023/0315(COD),European cross-border associations,LAGODINSKY Sergey (Greens/EFA),"ABADÍA JOVER Maravillas (EPP), WOLTERS Lara (S...",https://oeil.secure.europarl.europa.eu/oeil/en...
6,2023/0202(COD),General Data Protection Regulation: additional...,GREGOROVÁ Markéta (Greens/EFA),"VOSS Axel (EPP), VIGENIN Kristian (S&D), OZDOB...",https://oeil.secure.europarl.europa.eu/oeil/en...
7,2023/0089(COD),Company law: further expanding and upgrading t...,RADEV Emil (EPP),"LEITÃO-MARQUES Maria-Manuel (S&D), DZHAMBAZKI ...",https://oeil.secure.europarl.europa.eu/oeil/en...
8,2022/0411(COD),Making public capital markets in the Union mor...,SANT Alfred (S&D),"VAIDERE Inese (EPP), GRUFFAT Claude (Greens/EF...",https://oeil.secure.europarl.europa.eu/oeil/en...
9,2022/0408(COD),Harmonising certain aspects of insolvency law,RADEV Emil (EPP),"REPASI René (S&D), DIEPEVEEN Ton (PfE), PIPERE...",https://oeil.secure.europarl.europa.eu/oeil/en...


In [57]:
df_enter1.to_csv("df_enterprise.csv", index=False)
print("CSV file saved successfully!")

CSV file saved successfully!


### **Research and technological development and space**

In [58]:
url_research = "https://oeil.secure.europarl.europa.eu/oeil/en/search/export/XML?fullText.mode=EXACT_WORD&term=9th+term+2019+-+2024&subject=3.50+Research+and+technological+development+and+space&resultsOnly=true"

response = requests.get(url_research)
response.raise_for_status()

root = ET.fromstring(response.content)

data_research = []

for item in root.find("items").findall("item"):
    reference = item.findtext("reference", default="")
    title = item.findtext("title", default="")
    
    # Extract all rapporteurs 
    rapporteurs_elem = item.find("rapporteur")
    if rapporteurs_elem is not None:
        rapporteurs = [r.text for r in rapporteurs_elem.findall("rapporteur") if r.text]
    else:
        rapporteurs = ['NaN']

    rapporteurs_str = ", ".join(rapporteurs) if rapporteurs else ""

    data_research.append({
        "document_id": reference,
        "title": title,
        "rapporteurs": rapporteurs_str,
    })

df_research = pd.DataFrame(data_research)
print(df_research.head())

      document_id                                              title  \
0  2023/0207(COD)  Partnership for research and innovation in the...   
1  2023/0130(COD)  Supplementary protection certificate for medic...   
2  2023/0128(COD)  Supplementary protection certificate for plant...   
3  2023/0133(COD)                         Standard essential patents   
4  2023/0129(COD)  Compulsory licensing of patents in crisis situ...   

                   rapporteurs  
0           BORCHIA Paolo (ID)  
1           WÖLKEN Tiemo (S&D)  
2           WÖLKEN Tiemo (S&D)  
3        WALSMANN Marion (EPP)  
4  VÁZQUEZ LÁZARA Adrián (EPP)  


In [59]:
df_research['rapporteurs'] = df_research['rapporteurs'].replace('', np.nan) 
df_research

,document_id,title,rapporteurs
0,2023/0207(COD),Partnership for research and innovation in the...,BORCHIA Paolo (ID)
1,2023/0130(COD),Supplementary protection certificate for medic...,WÖLKEN Tiemo (S&D)
2,2023/0128(COD),Supplementary protection certificate for plant...,WÖLKEN Tiemo (S&D)
3,2023/0133(COD),Standard essential patents,WALSMANN Marion (EPP)
4,2023/0129(COD),Compulsory licensing of patents in crisis situ...,VÁZQUEZ LÁZARA Adrián (EPP)
...,...,...,...
59,COM(2024)0232,Opening of negotiations on the Design Law Trea...,NaN
60,SWD(2024)0176,EU Space Programme user uptake status,NaN
61,SWD(2024)0162,Commission ex-ante analysis on the relevance o...,NaN
62,SWD(2024)0160,Solar energy joint research and innovation age...,NaN


In [60]:
#removing nan
df_research1 = df_research.dropna()
df_research1.reset_index(drop=True, inplace= True)
df_research1

,document_id,title,rapporteurs
0,2023/0207(COD),Partnership for research and innovation in the...,BORCHIA Paolo (ID)
1,2023/0130(COD),Supplementary protection certificate for medic...,WÖLKEN Tiemo (S&D)
2,2023/0128(COD),Supplementary protection certificate for plant...,WÖLKEN Tiemo (S&D)
3,2023/0133(COD),Standard essential patents,WALSMANN Marion (EPP)
4,2023/0129(COD),Compulsory licensing of patents in crisis situ...,VÁZQUEZ LÁZARA Adrián (EPP)
5,2023/0127(COD),Unitary supplementary certificate for medicina...,WÖLKEN Tiemo (S&D)
6,2023/0126(COD),Unitary supplementary protection certificate f...,WÖLKEN Tiemo (S&D)
7,2022/0392(COD),Industrial property: legal protection of desig...,LEBRETON Gilles (ID)
8,2022/0391(COD),Industrial property: protection of Community d...,LEBRETON Gilles (ID)
9,2022/0115(COD),Geographical indication protection for craft a...,WALSMANN Marion (EPP)


#### Find shadow rapporteurs and document url

In [61]:
df_test = df_research1.copy()
df_test = df_test[:1]
df_test

,document_id,title,rapporteurs
0,2023/0207(COD),Partnership for research and innovation in the...,BORCHIA Paolo (ID)


In [62]:
def get_shadow_and_link(reference):
    base_url = "https://oeil.secure.europarl.europa.eu/oeil/en/procedure-file?reference="
    url_research = base_url + reference

    response = requests.get(url_research)
    if response.status_code != 200:
        print(f"Failed to fetch: {url_research}")
        return None

    soup = BeautifulSoup(response.content, "html.parser")

    try:
        #finding shadow rapporteurs
        shadow_section = soup.find("div", id="collapseShadowRapporteur")
        if not shadow_section:
            return None
        
        shadow_names = []
        for a in shadow_section.find_all("a", class_= "rapporteur"):
            span = a.find("span")
            #print("span", span)
            if span:
                shadow_names.append(span.get_text(strip=True))
                #print("shadow names:", shadow_names)

        shadow_str = ", ".join(shadow_names) if shadow_names else None

        #finding summary link
        section6 = soup.find("div", id="section6")
        oeil_link = None  # <-- initialize here

        if section6:
            ec_span = section6.find("span", string="European Commission")
            if ec_span:
                accordion_item = ec_span.find_parent("li", class_="erpl_accordion-item")
                if accordion_item:
                    summary_links = accordion_item.find_all("a", href=True)
                    for a in summary_links:
                        href = a.get("href")
                        if href and href.startswith("/oeil"):
                            oeil_link = "https://oeil.secure.europarl.europa.eu" + href
                            break  

        return shadow_str, oeil_link



    
    except Exception as e:
        print(f"Error parsing {reference}: {e}")
        return None

df_research1[["shadow_rapporteurs","summary_link"]] = df_research1["document_id"].apply(
    lambda ref: pd.Series(get_shadow_and_link(ref))
)

/var/folders/c_/q2yk81hd0zj7jnx_b5275pzh0000gn/T/ipykernel_7561/1379865773.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_research1[["shadow_rapporteurs","summary_link"]] = df_research1["document_id"].apply(
/var/folders/c_/q2yk81hd0zj7jnx_b5275pzh0000gn/T/ipykernel_7561/1379865773.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_research1[["shadow_rapporteurs","summary_link"]] = df_research1["document_id"].apply(


In [63]:
df_research1

,document_id,title,rapporteurs,shadow_rapporteurs,summary_link
0,2023/0207(COD),Partnership for research and innovation in the...,BORCHIA Paolo (ID),"CARVALHO Maria da Graça (EPP), GÁLVEZ Lina (S&...",https://oeil.secure.europarl.europa.eu/oeil/en...
1,2023/0130(COD),Supplementary protection certificate for medic...,WÖLKEN Tiemo (S&D),"ZARZALEJOS Javier (EPP), ZŁOTOWSKI Kosma (ECR)...",https://oeil.secure.europarl.europa.eu/oeil/en...
2,2023/0128(COD),Supplementary protection certificate for plant...,WÖLKEN Tiemo (S&D),"ZARZALEJOS Javier (EPP), ZŁOTOWSKI Kosma (ECR)...",https://oeil.secure.europarl.europa.eu/oeil/en...
3,2023/0133(COD),Standard essential patents,WALSMANN Marion (EPP),"WÖLKEN Tiemo (S&D), ZŁOTOWSKI Kosma (ECR), FUR...",https://oeil.secure.europarl.europa.eu/oeil/en...
4,2023/0129(COD),Compulsory licensing of patents in crisis situ...,VÁZQUEZ LÁZARA Adrián (EPP),"WÖLKEN Tiemo (S&D), ZŁOTOWSKI Kosma (ECR), FAR...",https://oeil.secure.europarl.europa.eu/oeil/en...
5,2023/0127(COD),Unitary supplementary certificate for medicina...,WÖLKEN Tiemo (S&D),"ZARZALEJOS Javier (EPP), ZŁOTOWSKI Kosma (ECR)...",https://oeil.secure.europarl.europa.eu/oeil/en...
6,2023/0126(COD),Unitary supplementary protection certificate f...,WÖLKEN Tiemo (S&D),"ZARZALEJOS Javier (EPP), ZŁOTOWSKI Kosma (ECR)...",https://oeil.secure.europarl.europa.eu/oeil/en...
7,2022/0392(COD),Industrial property: legal protection of desig...,LEBRETON Gilles (ID),"MANDERS Antonius (EPP), GARCÍA DEL BLANCO Ibán...",None
8,2022/0391(COD),Industrial property: protection of Community d...,LEBRETON Gilles (ID),"MANDERS Antonius (EPP), GARCÍA DEL BLANCO Ibán...",None
9,2022/0115(COD),Geographical indication protection for craft a...,WALSMANN Marion (EPP),"GARCÍA DEL BLANCO Ibán (S&D), VÁZQUEZ LÁZARA A...",https://oeil.secure.europarl.europa.eu/oeil/en...


In [64]:
df_research1.to_csv("df_research.csv", index=False)
print("CSV file saved successfully!")

CSV file saved successfully!


### **Energy policy**

In [65]:
url_energy = "https://oeil.secure.europarl.europa.eu/oeil/en/search/export/XML?fullText.mode=EXACT_WORD&term=9th+term+2019+-+2024&subject=3.60+Energy+policy&resultsOnly=true"

response = requests.get(url_energy)
response.raise_for_status()

root = ET.fromstring(response.content)

data_energy = []

for item in root.find("items").findall("item"):
    reference = item.findtext("reference", default="")
    title = item.findtext("title", default="")
    
    # Extract all rapporteurs 
    rapporteurs_elem = item.find("rapporteur")
    if rapporteurs_elem is not None:
        rapporteurs = [r.text for r in rapporteurs_elem.findall("rapporteur") if r.text]
    else:
        rapporteurs = ['NaN']

    rapporteurs_str = ", ".join(rapporteurs) if rapporteurs else ""

    data_energy.append({
        "document_id": reference,
        "title": title,
        "rapporteurs": rapporteurs_str,
    })

df_energy = pd.DataFrame(data_energy)
print(df_energy.head())

       document_id                                              title  \
0   2024/0148(COD)  EU/Euratom Agreement on the interpretation and...   
1  2023/0077B(COD)                  Union’s electricity market design   
2   2023/0362(COD)  Amending certain road transport and aviation D...   
3   2023/0272(COD)  Mercury: dental amalgam and other mercury-adde...   
4  2023/0077A(COD)                  Union’s electricity market design   

                                      rapporteurs  
0  BUDKA Borys (EPP), CAVAZZINI Anna (Greens/EFA)  
1                  GONZÁLEZ CASARES Nicolás (S&D)  
2                    OETJEN Jan-Christoph (Renew)  
3                           MORTLER Marlene (EPP)  
4                  GONZÁLEZ CASARES Nicolás (S&D)  


In [66]:
df_energy['rapporteurs'] = df_energy['rapporteurs'].replace('', np.nan) 
df_energy

,document_id,title,rapporteurs
0,2024/0148(COD),EU/Euratom Agreement on the interpretation and...,"BUDKA Borys (EPP), CAVAZZINI Anna (Greens/EFA)"
1,2023/0077B(COD),Union’s electricity market design,GONZÁLEZ CASARES Nicolás (S&D)
2,2023/0362(COD),Amending certain road transport and aviation D...,OETJEN Jan-Christoph (Renew)
3,2023/0272(COD),Mercury: dental amalgam and other mercury-adde...,MORTLER Marlene (EPP)
4,2023/0077A(COD),Union’s electricity market design,GONZÁLEZ CASARES Nicolás (S&D)
...,...,...,...
87,2020/2917(RPS),Decision to raise no objections to the draft C...,CANFIN Pascal (Renew)
88,COM(2024)0181,Implementation of the work under the nuclear d...,NaN
89,COM(2023)0793,Approving a Commission Regulation (Euratom) on...,NaN
90,SWD(2024)0159,Towards a roadmap for accelerating the deploym...,NaN


In [67]:
#removing nan
df_energy1 = df_energy.dropna()
df_energy1.reset_index(drop=True, inplace= True)
df_energy1

,document_id,title,rapporteurs
0,2024/0148(COD),EU/Euratom Agreement on the interpretation and...,"BUDKA Borys (EPP), CAVAZZINI Anna (Greens/EFA)"
1,2023/0077B(COD),Union’s electricity market design,GONZÁLEZ CASARES Nicolás (S&D)
2,2023/0362(COD),Amending certain road transport and aviation D...,OETJEN Jan-Christoph (Renew)
3,2023/0272(COD),Mercury: dental amalgam and other mercury-adde...,MORTLER Marlene (EPP)
4,2023/0077A(COD),Union’s electricity market design,GONZÁLEZ CASARES Nicolás (S&D)
5,2023/0076(COD),Wholesale energy market: Union’s protection ag...,CARVALHO Maria da Graça (EPP)
6,2022/0164(COD),REPowerEU chapters in recovery and resilience ...,"MUREŞAN Siegfried (EPP), GARDIAZABAL RUBIAL Ei..."
7,2022/0090(COD),Security of gas supply and conditions for acce...,BUŞOI Cristian-Silviu (EPP)
8,2021/0425(COD),Gas and hydrogen markets directive (common rules),GEIER Jens (S&D)
9,2021/0424(COD),Gas and hydrogen markets regulation,BUZEK Jerzy (EPP)


#### Find shadow rapporteurs and document url

In [68]:
df_test = df_energy1.copy()
df_test = df_test[:1]
df_test

,document_id,title,rapporteurs
0,2024/0148(COD),EU/Euratom Agreement on the interpretation and...,"BUDKA Borys (EPP), CAVAZZINI Anna (Greens/EFA)"


In [69]:
def get_shadow_and_link(reference):
    base_url = "https://oeil.secure.europarl.europa.eu/oeil/en/procedure-file?reference="
    url_energy = base_url + reference

    response = requests.get(url_energy)
    if response.status_code != 200:
        print(f"Failed to fetch: {url_energy}")
        return None

    soup = BeautifulSoup(response.content, "html.parser")

    try:
        #finding shadow rapporteurs
        shadow_section = soup.find("div", id="collapseShadowRapporteur")
        if not shadow_section:
            return None
        
        shadow_names = []
        for a in shadow_section.find_all("a", class_= "rapporteur"):
            span = a.find("span")
            #print("span", span)
            if span:
                shadow_names.append(span.get_text(strip=True))
                #print("shadow names:", shadow_names)

        shadow_str = ", ".join(shadow_names) if shadow_names else None

        #finding summary link
        section6 = soup.find("div", id="section6")
        oeil_link = None  # <-- initialize here

        if section6:
            ec_span = section6.find("span", string="European Commission")
            if ec_span:
                accordion_item = ec_span.find_parent("li", class_="erpl_accordion-item")
                if accordion_item:
                    summary_links = accordion_item.find_all("a", href=True)
                    for a in summary_links:
                        href = a.get("href")
                        if href and href.startswith("/oeil"):
                            oeil_link = "https://oeil.secure.europarl.europa.eu" + href
                            break  

        return shadow_str, oeil_link



    
    except Exception as e:
        print(f"Error parsing {reference}: {e}")
        return None

df_energy1[["shadow_rapporteurs","summary_link"]] = df_energy1["document_id"].apply(
    lambda ref: pd.Series(get_shadow_and_link(ref))
)

/var/folders/c_/q2yk81hd0zj7jnx_b5275pzh0000gn/T/ipykernel_7561/694176302.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_energy1[["shadow_rapporteurs","summary_link"]] = df_energy1["document_id"].apply(
/var/folders/c_/q2yk81hd0zj7jnx_b5275pzh0000gn/T/ipykernel_7561/694176302.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_energy1[["shadow_rapporteurs","summary_link"]] = df_energy1["document_id"].apply(


In [70]:
df_energy1

,document_id,title,rapporteurs,shadow_rapporteurs,summary_link
0,2024/0148(COD),EU/Euratom Agreement on the interpretation and...,"BUDKA Borys (EPP), CAVAZZINI Anna (Greens/EFA)","CASPARY Daniel (EPP), NICA Dan (S&D), GÁLVEZ L...",https://oeil.secure.europarl.europa.eu/oeil/en...
1,2023/0077B(COD),Union’s electricity market design,GONZÁLEZ CASARES Nicolás (S&D),"CARVALHO Maria da Graça (EPP), PETERSEN Morten...",None
2,2023/0362(COD),Amending certain road transport and aviation D...,OETJEN Jan-Christoph (Renew),"MARINESCU Marian-Jean (EPP), CERDAS Sara (S&D)...",https://oeil.secure.europarl.europa.eu/oeil/en...
3,2023/0272(COD),Mercury: dental amalgam and other mercury-adde...,MORTLER Marlene (EPP),"FRITZON Heléne (S&D), AMALRIC Catherine (Renew...",https://oeil.secure.europarl.europa.eu/oeil/en...
4,2023/0077A(COD),Union’s electricity market design,GONZÁLEZ CASARES Nicolás (S&D),"CARVALHO Maria da Graça (EPP), PETERSEN Morten...",https://oeil.secure.europarl.europa.eu/oeil/en...
5,2023/0076(COD),Wholesale energy market: Union’s protection ag...,CARVALHO Maria da Graça (EPP),"TOIA Patrizia (S&D), GAMON Claudia (Renew), DA...",https://oeil.secure.europarl.europa.eu/oeil/en...
6,2022/0164(COD),REPowerEU chapters in recovery and resilience ...,"MUREŞAN Siegfried (EPP), GARDIAZABAL RUBIAL Ei...","MAVRIDES Costas (S&D), URTASUN Ernest (Greens/...",https://oeil.secure.europarl.europa.eu/oeil/en...
7,2022/0090(COD),Security of gas supply and conditions for acce...,BUŞOI Cristian-Silviu (EPP),NaN,NaN
8,2021/0425(COD),Gas and hydrogen markets directive (common rules),GEIER Jens (S&D),"BUZEK Jerzy (EPP), GAMON Claudia (Renew), CORR...",https://oeil.secure.europarl.europa.eu/oeil/en...
9,2021/0424(COD),Gas and hydrogen markets regulation,BUZEK Jerzy (EPP),"TOIA Patrizia (S&D), GROŠELJ Klemen (Renew), T...",https://oeil.secure.europarl.europa.eu/oeil/en...


In [71]:
df_energy1.to_csv("df_energy.csv", index=False)
print("CSV file saved successfully!")

CSV file saved successfully!


### **Environmental policy**

In [72]:
url_environ = "https://oeil.secure.europarl.europa.eu/oeil/en/search/export/XML?fullText.mode=EXACT_WORD&term=9th+term+2019+-+2024&subject=3.70+Environmental+policy&resultsOnly=true"

response = requests.get(url_environ)
response.raise_for_status()

root = ET.fromstring(response.content)

data_environ = []

for item in root.find("items").findall("item"):
    reference = item.findtext("reference", default="")
    title = item.findtext("title", default="")
    
    # Extract all rapporteurs 
    rapporteurs_elem = item.find("rapporteur")
    if rapporteurs_elem is not None:
        rapporteurs = [r.text for r in rapporteurs_elem.findall("rapporteur") if r.text]
    else:
        rapporteurs = ['NaN']

    rapporteurs_str = ", ".join(rapporteurs) if rapporteurs else ""

    data_environ.append({
        "document_id": reference,
        "title": title,
        "rapporteurs": rapporteurs_str,
    })

df_environ = pd.DataFrame(data_environ)
print(df_environ.head())

      document_id                                              title  \
0  2023/0453(COD)  Common data platform on chemicals, establishin...   
1  2023/0455(COD)  Chemicals: re-attribution of scientific and te...   
2  2023/0454(COD)  Restriction of the use of certain hazardous su...   
3  2023/0378(COD)  Protective measures against pests of plants: m...   
4  2023/0373(COD)  Preventing plastic pellet losses to reduce mic...   

               rapporteurs  
0  TSIODRAS Dimitris (EPP)  
1  TSIODRAS Dimitris (EPP)  
2  TSIODRAS Dimitris (EPP)  
3     AGUILERA Clara (S&D)  
4        LUENA César (S&D)  


In [73]:
df_environ['rapporteurs'] = df_environ['rapporteurs'].replace('', np.nan) 
df_environ

,document_id,title,rapporteurs
0,2023/0453(COD),"Common data platform on chemicals, establishin...",TSIODRAS Dimitris (EPP)
1,2023/0455(COD),Chemicals: re-attribution of scientific and te...,TSIODRAS Dimitris (EPP)
2,2023/0454(COD),Restriction of the use of certain hazardous su...,TSIODRAS Dimitris (EPP)
3,2023/0378(COD),Protective measures against pests of plants: m...,AGUILERA Clara (S&D)
4,2023/0373(COD),Preventing plastic pellet losses to reduce mic...,LUENA César (S&D)
...,...,...,...
415,SWD(2024)0172,Annual report on taxation 2024 - Review of tax...,NaN
416,SWD(2024)0162,Commission ex-ante analysis on the relevance o...,NaN
417,SWD(2024)0147,Supporting Indoor Air Quality,NaN
418,C(2024)3817,European citizens’ initiative: ‘Air-Quotas’. C...,NaN


In [74]:
#removing nan
df_environ1 = df_environ.dropna()
df_environ1.reset_index(drop=True, inplace= True)
df_environ1

,document_id,title,rapporteurs
0,2023/0453(COD),"Common data platform on chemicals, establishin...",TSIODRAS Dimitris (EPP)
1,2023/0455(COD),Chemicals: re-attribution of scientific and te...,TSIODRAS Dimitris (EPP)
2,2023/0454(COD),Restriction of the use of certain hazardous su...,TSIODRAS Dimitris (EPP)
3,2023/0378(COD),Protective measures against pests of plants: m...,AGUILERA Clara (S&D)
4,2023/0373(COD),Preventing plastic pellet losses to reduce mic...,LUENA César (S&D)
...,...,...,...
131,2022/2524(RPS),Objection pursuant to Rule 112(2): Maximum res...,"PIETIKÄINEN Sirpa (EPP), ARENA Maria (S&D), PA..."
132,2020/2917(RPS),Decision to raise no objections to the draft C...,CANFIN Pascal (Renew)
133,2020/2898(RPS),Decision to raise no objections to the draft C...,CANFIN Pascal (Renew)
134,2019/2949(RPS),Resolution on the draft Commission regulation ...,"ARENA Maria (S&D), HOJSÍK Martin (Renew), EICK..."


#### Find shadow rapporteurs and document url

In [75]:
df_test = df_environ1.copy()
df_test = df_test[:1]
df_test

,document_id,title,rapporteurs
0,2023/0453(COD),"Common data platform on chemicals, establishin...",TSIODRAS Dimitris (EPP)


In [76]:
def get_shadow_and_link(reference):
    base_url = "https://oeil.secure.europarl.europa.eu/oeil/en/procedure-file?reference="
    url_environ = base_url + reference

    response = requests.get(url_environ)
    if response.status_code != 200:
        print(f"Failed to fetch: {url_environ}")
        return None

    soup = BeautifulSoup(response.content, "html.parser")

    try:
        #finding shadow rapporteurs
        shadow_section = soup.find("div", id="collapseShadowRapporteur")
        if not shadow_section:
            return None
        
        shadow_names = []
        for a in shadow_section.find_all("a", class_= "rapporteur"):
            span = a.find("span")
            #print("span", span)
            if span:
                shadow_names.append(span.get_text(strip=True))
                #print("shadow names:", shadow_names)

        shadow_str = ", ".join(shadow_names) if shadow_names else None

        #finding summary link
        section6 = soup.find("div", id="section6")
        oeil_link = None  # <-- initialize here

        if section6:
            ec_span = section6.find("span", string="European Commission")
            if ec_span:
                accordion_item = ec_span.find_parent("li", class_="erpl_accordion-item")
                if accordion_item:
                    summary_links = accordion_item.find_all("a", href=True)
                    for a in summary_links:
                        href = a.get("href")
                        if href and href.startswith("/oeil"):
                            oeil_link = "https://oeil.secure.europarl.europa.eu" + href
                            break  

        return shadow_str, oeil_link



    
    except Exception as e:
        print(f"Error parsing {reference}: {e}")
        return None

df_environ1[["shadow_rapporteurs","summary_link"]] = df_environ1["document_id"].apply(
    lambda ref: pd.Series(get_shadow_and_link(ref))
)

/var/folders/c_/q2yk81hd0zj7jnx_b5275pzh0000gn/T/ipykernel_7561/158641509.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_environ1[["shadow_rapporteurs","summary_link"]] = df_environ1["document_id"].apply(
/var/folders/c_/q2yk81hd0zj7jnx_b5275pzh0000gn/T/ipykernel_7561/158641509.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_environ1[["shadow_rapporteurs","summary_link"]] = df_environ1["document_id"].apply(


In [77]:
df_environ1

,document_id,title,rapporteurs,shadow_rapporteurs,summary_link
0,2023/0453(COD),"Common data platform on chemicals, establishin...",TSIODRAS Dimitris (EPP),"CLERGEAU Christophe (S&D), TIMGREN Beatrice (E...",https://oeil.secure.europarl.europa.eu/oeil/en...
1,2023/0455(COD),Chemicals: re-attribution of scientific and te...,TSIODRAS Dimitris (EPP),"CLERGEAU Christophe (S&D), TIMGREN Beatrice (E...",https://oeil.secure.europarl.europa.eu/oeil/en...
2,2023/0454(COD),Restriction of the use of certain hazardous su...,TSIODRAS Dimitris (EPP),"CLERGEAU Christophe (S&D), TIMGREN Beatrice (E...",https://oeil.secure.europarl.europa.eu/oeil/en...
3,2023/0378(COD),Protective measures against pests of plants: m...,AGUILERA Clara (S&D),"BUDA Daniel (EPP), MÜLLER Ulrike (Renew), RUIS...",https://oeil.secure.europarl.europa.eu/oeil/en...
4,2023/0373(COD),Preventing plastic pellet losses to reduce mic...,LUENA César (S&D),"SOMMEN Liesbet (EPP), BONTE Barbara (PfE), FIO...",https://oeil.secure.europarl.europa.eu/oeil/en...
...,...,...,...,...,...
131,2022/2524(RPS),Objection pursuant to Rule 112(2): Maximum res...,"PIETIKÄINEN Sirpa (EPP), ARENA Maria (S&D), PA...",NaN,NaN
132,2020/2917(RPS),Decision to raise no objections to the draft C...,CANFIN Pascal (Renew),NaN,NaN
133,2020/2898(RPS),Decision to raise no objections to the draft C...,CANFIN Pascal (Renew),NaN,NaN
134,2019/2949(RPS),Resolution on the draft Commission regulation ...,"ARENA Maria (S&D), HOJSÍK Martin (Renew), EICK...",NaN,NaN


In [78]:
df_environ1.to_csv("df_environmental.csv", index=False)
print("CSV file saved successfully!")

CSV file saved successfully!
